<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 1 — Pipeline de preprocesamiento de texto</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De texto crudo a términos limpios · NLTK + spaCy (es_core_news_sm)</div></div>

## Reglas de entrega

- **Repo:** suban este notebook (ya ejecutado, con sus salidas) a su repositorio de GitHub Classroom de la organización `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio:** si usaron IA en cualquier parte, declárenlo en ese archivo (qué herramienta, para qué celda, qué les dio y qué cambiaron). No declararlo se considera falta de honestidad académica.
- **Defensa oral (eliminatoria):** se les preguntará por *cualquier* celda. "Lo generó la IA" o "lo dijo el profesor" no son justificaciones válidas. Si no pueden explicar su código, no hay calificación.
- **Penalizaciones por entrega tardía:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Las celdas marcadas con `# TODO` y las preguntas en **negritas** son lo que se evalúa. El resto es andamiaje que ya viene resuelto para que enfoquen su tiempo en lo que importa.


## Objetivo

Construir el pipeline que convierte una noticia cruda en una lista de términos limpios, tomando **decisiones justificadas** en cada paso (no recetas memorizadas). El corpus que limpien aquí es el mismo que usarán en el **Lab 2** para construir su motor de búsqueda, así que las decisiones de hoy se arrastran al resto de la unidad.


## 0 · Entorno

Ejecuten una sola vez. Si están en Colab, descomenten la primera línea.

In [1]:
import re, unicodedata, json
from collections import Counter
import pandas as pd

import nltk
nltk.download('punkt', quiet=True)
from nltk.stem import SnowballStemmer

import spacy
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

print('Entorno listo.')

[+] Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
Entorno listo.


## Corpus

Noticias chiapanecas (sintéticas) con ruido real: HTML, URLs, emojis, acentos faltantes, dobles espacios. **No editen el corpus.**

In [2]:
corpus_crudo = [
    {"id": "d01", "titulo": "Lluvias provocan inundaciones en Tuxtla",
     "texto": "Las  fuertes lluvias   provocaron inundaciones en varias colonias del sur de Tuxtla Gutierrez 😟. "
              "Proteccion Civil pidio a la poblacion no cruzar las calles anegadas. Mas info en https://chiapasparalelo.com/nota1 ."},
    {"id": "d02", "titulo": "Crisis hidrica golpea la region",
     "texto": "La crisis hidrica se agrava: el desabasto del liquido vital afecta a miles de familias en la zona alta. "
              "Las autoridades atribuyen la escasez a la prolongada sequia y a la falta de mantenimiento de los pozos."},
    {"id": "d03", "titulo": "Cafe de Chiapas rompe record de exportacion",
     "texto": "<p>El <b>cafe</b> de Chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en Europa y Asia.</p> "
              "Los productores de la Sierra celebran precios al alza."},
    {"id": "d04", "titulo": "Sequia afecta cultivos de maiz",
     "texto": "La sequia afecta gravemente los cultivos de maiz y frijol en la region fronteriza. "
              "Los agricultores reportan perdidas de hasta el 40% y piden apoyos al gobierno estatal."},
    {"id": "d05", "titulo": "Turismo crece en el Canon del Sumidero",
     "texto": "El Canon del Sumidero recibio mas de 200 mil visitantes durante la temporada. "
              "Los recorridos en lancha y el avistamiento de fauna son los principales atractivos del parque nacional. #Chiapas"},
    {"id": "d06", "titulo": "Sismo de magnitud 5.1 frente a las costas",
     "texto": "Un sismo de magnitud 5.1 se registro frente a las costas de Chiapas la madrugada del martes. "
              "No se reportaron danos ni victimas, informo el Servicio Sismologico Nacional."},
    {"id": "d07", "titulo": "UPCh inaugura laboratorio de IA",
     "texto": "La Universidad Politecnica de Chiapas inauguro un nuevo laboratorio de inteligencia artificial "
              "equipado con GPUs para proyectos de aprendizaje automatico y vision por computadora. Visita https://upchiapas.edu.mx ."},
    {"id": "d08", "titulo": "Repunta la produccion de cacao",
     "texto": "La produccion de cacao en la region del Soconusco repunto este año tras varios ciclos a la baja. "
              "Cooperativas locales apuestan por el chocolate artesanal de origen para mercados premium."},
    {"id": "d09", "titulo": "San Cristobal, destino cultural",
     "texto": "San Cristobal de las Casas se consolida como destino cultural: sus mercados, iglesias y cafeterias "
              "atraen a viajeros de todo el mundo. La gastronomia y el textil artesanal son protagonistas."},
    {"id": "d10", "titulo": "Avanza obra de infraestructura carretera",
     "texto": "Avanza la rehabilitacion de la carretera que conecta Tuxtla con la costa. "
              "La obra busca reducir tiempos de traslado y mejorar la seguridad vial para miles de automovilistas."},
    {"id": "d11", "titulo": "Alertan por casos de dengue",
     "texto": "La Secretaria de Salud alerto por un repunte de casos de dengue en municipios de tierra caliente. "
              "Pide a la poblacion eliminar criaderos de mosco y acudir al medico ante fiebre alta. 🦟"},
    {"id": "d12", "titulo": "Feria celebra el cafe y el cacao",
     "texto": "La feria regional celebro el cafe y el cacao chiapaneco con catas, musica y venta directa de productores. "
              "Miles de asistentes recorrieron los stands durante el fin de semana."},
    {"id": "d13", "titulo": "Restablecen servicio de agua potable",
     "texto": "El servicio de agua potable se restablecera de forma escalonada en las colonias afectadas por la falla en la red. "
              "El organismo operador pidio a los usuarios almacenar agua de manera responsable."},
    {"id": "d14", "titulo": "Estudiantes ganan concurso de robotica",
     "texto": "Estudiantes de ingenieria de Tuxtla ganaron el primer lugar en un concurso nacional de robotica 🤖 "
              "con un brazo robotico de bajo costo. El equipo representara a Mexico en una competencia internacional."},
]
print(f"{len(corpus_crudo)} documentos cargados.")

14 documentos cargados.


In [3]:
df = pd.DataFrame(corpus_crudo)
df['n_chars'] = df['texto'].str.len()
df[['id', 'titulo', 'n_chars']]

## 1 · Cargar y explorar

Antes de limpiar, hay que **mirar** los datos. Una limpieza a ciegas destruye señal sin que se den cuenta.


**1.a** Estadísticas de longitud. Completen los cálculos.

In [4]:
# TODO: reporten el numero de documentos, la longitud media en caracteres,
#       y la longitud media en PALABRAS (split ingenuo por espacios).
#       Pista: df['texto'].str.split().str.len()
n_docs = len(df)
longitud_media_chars = df['n_chars'].mean()
longitud_media_palabras = df['texto'].str.split().str.len().mean()
print(f"Número de documentos: {n_docs}")
print(f"Longitud media en caracteres: {longitud_media_chars:.2f}")
print(f"Longitud media en palabras (split ingenuo): {longitud_media_palabras:.2f}")


Número de documentos: 14
Longitud media en caracteres: 189.07
Longitud media en palabras (split ingenuo): 30.21


**1.b** Detección de ruido. Les damos un detector de URLs ya hecho como ejemplo. **Completen** los detectores de etiquetas HTML y de emojis, y reporten en qué documentos aparece cada tipo de ruido.

In [5]:
RE_URL  = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')
RE_EMOJI = re.compile(r'[\U00010000-\U0010FFFF\u2600-\u27BF]')

for fila in corpus_crudo:
    texto = fila['texto']
    urls = RE_URL.findall(texto)
    htmls = RE_HTML.findall(texto)
    emojis = RE_EMOJI.findall(texto)
    
    if urls or htmls or emojis:
        print(f"Documento {fila['id']}:")
        if urls:
            print('  -> URL:', urls)
        if htmls:
            print('  -> HTML:', htmls)
        if emojis:
            print('  -> Emojis:', emojis)


Documento d01:
  -> URL: ['https://chiapasparalelo.com/nota1']
  -> Emojis: ['😟']
Documento d03:
  -> HTML: ['<p>', '<b>', '</b>', '</p>']
Documento d07:
  -> URL: ['https://upchiapas.edu.mx']
Documento d11:
  -> Emojis: ['🦟']
Documento d14:
  -> Emojis: ['🤖']


> **Pregunta (defensa):** de los tres tipos de ruido, ¿cuál *podría* ser señal útil en algún dominio y por qué? Respondan en la celda de abajo.


_Su respuesta:_
- **Emojis:** Son extremadamente útiles en tareas de análisis de sentimientos y minería de opiniones en redes sociales o comentarios de usuarios (por ejemplo, para detectar sarcasmo, enojo o felicidad de forma instantánea), ya que a menudo condensan la emoción del texto directamente sin requerir análisis sintáctico complejo.
- **URLs:** Son valiosas para la clasificación de spam, detección de enlaces maliciosos (ciberseguridad), minería de grafos web y sistemas de recomendación, o para verificar la reputación y veracidad de las fuentes en periodismo de datos.
- **HTML:** Es útil cuando se realiza web scraping o extracción estructurada de información, puesto que las etiquetas como `<b>`, `<h1>`, o `<table>` indican la importancia relativa de las palabras o el formato tabular del contenido que se desea indexar o extraer.


## 2 · Tokenizar y normalizar

Tokenizar = decidir dónde empieza y termina cada unidad. Normalizar = que dos formas superficiales de la misma palabra colapsen en el mismo término.


**2.a** Comparen el `split` ingenuo contra un tokenizador real (spaCy). Observen las diferencias en puntuación y contracciones.

In [6]:
ejemplo = corpus_crudo[0]['texto']

ingenuo = ejemplo.split()           # tokenizacion por espacios
print('Ingenuo  :', ingenuo[:12])

# Tokenizar 'ejemplo' con spaCy (nlp) y mostrar los primeros 12 tokens.
doc = nlp(ejemplo)
spacy_tokens = [token.text for token in doc]
print('spaCy    :', spacy_tokens[:12])

# Diferencias comentadas:
# 1. Separación de puntuación y emojis: spaCy aísla los signos de puntuación y emojis como tokens independientes (por ejemplo, separa el emoji '😟' y el punto '.' en tokens individuales), mientras que el split ingenuo los mantiene pegados a la palabra anterior ('Tuxtla' y '.' vs 'Tuxtla.').
# 2. Manejo de espaciados múltiples: El split ingenuo colapsa los dobles espacios de forma automática, mientras que la tokenización por defecto de spaCy puede generar tokens de espacio vacío o manejarlos de forma diferente según la configuración de su pipeline.


Ingenuo  : ['Las', 'fuertes', 'lluvias', 'provocaron', 'inundaciones', 'en', 'varias', 'colonias', 'del', 'sur', 'de', 'Tuxtla']
spaCy    : ['Las', ' ', 'fuertes', 'lluvias', '  ', 'provocaron', 'inundaciones', 'en', 'varias', 'colonias', 'del', 'sur']


**2.b** Función de normalización. Debe: pasar a minúsculas, aplicar **Unicode NFC**, quitar HTML, URLs y colapsar espacios. **Decisión clave:** ¿quitan o conservan acentos? Impleméntenlo como un parámetro y justifiquen su elección por defecto más abajo.

In [7]:
def normalizar(texto, quitar_acentos=False):
    """Devuelve el texto normalizado (aun como string, sin tokenizar)."""
    # Convertir a minúsculas
    texto = texto.lower()
    # Eliminar HTML
    texto = re.sub(r'<[^>]+>', '', texto)
    # Eliminar URLs
    texto = re.sub(r'https?://\S+', '', texto)
    # Si quitar_acentos=True, eliminar diacríticos
    if quitar_acentos:
        texto = unicodedata.normalize('NFD', texto)
        texto = "".join([c for c in texto if unicodedata.category(c) != 'Mn'])
    # Normalización Unicode NFC
    texto = unicodedata.normalize('NFC', texto)
    # Colapsar espacios múltiples y recortar extremos
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# prueba rapida
print(normalizar(corpus_crudo[2]['texto']))


el cafe de chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en europa y asia. los productores de la sierra celebran precios al alza.


> **Decisión documentada:** ¿quitar acentos por defecto, sí o no? Den **un** argumento a favor y **uno** en contra, y digan cuál pesa más *para un buscador* donde el usuario casi nunca escribe acentos.


_Su decisión y justificación:_
**Decisión:** Sí, quitar acentos por defecto (`quitar_acentos=True`).
- **Argumento a favor:** En los motores de búsqueda (recuperación de información), los usuarios finales casi nunca escriben acentos al realizar consultas rápidas en teclados físicos o móviles. Quitar los acentos de raíz en el pipeline incrementa la exhaustividad (recall), permitiendo que la búsqueda de "cafe" devuelva documentos que contienen "café" de manera directa.
- **Argumento en contra:** Se pierde precisión semántica y distinción morfológica fina, provocando la colisión de palabras distintas (por ejemplo, "público", "publico" y "publicó" colapsan al mismo término "publico").
- **Conclusión:** Para un buscador general o de noticias de este tipo, la ganancia en exhaustividad (que el usuario encuentre lo que busca sin preocuparse por la ortografía) supera por mucho la pérdida de precisión por ambigüedad semántica.


## 3 · Stopwords con criterio

Filtrar palabras de función reduce ruido y dimensionalidad… pero la lista que viene "de fábrica" puede borrar señal de su dominio.


In [8]:
stopwords_es = nlp.Defaults.stop_words
print('Total de stopwords de spaCy (es):', len(stopwords_es))
print(sorted(list(stopwords_es))[:30])

Total de stopwords de spaCy (es): 521
['a', 'acuerdo', 'adelante', 'ademas', 'además', 'afirmó', 'agregó', 'ahi', 'ahora', 'ahí', 'al', 'algo', 'alguna', 'algunas', 'alguno', 'algunos', 'algún', 'alli', 'allí', 'alrededor', 'ambos', 'ante', 'anterior', 'antes', 'apenas', 'aproximadamente', 'aquel', 'aquella', 'aquellas', 'aquello']


**3.a** Inspeccionen la lista y encuentren **al menos una** palabra que, en el dominio de noticias de Chiapas, *sí* podría ser señal (piensen en negaciones, intensificadores o términos que cambian el sentido). Construyan su propia lista `MIS_STOPWORDS` quitándola(s).

In [9]:
# Conservamos 'no', 'sin' y 'contra' porque en el dominio de noticias la negación y oposición cambian completamente el sentido.
CONSERVAR = {'no', 'sin', 'contra'}
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR
print(f"Stopwords originales: {len(stopwords_es)}")
print(f"Stopwords personalizadas: {len(MIS_STOPWORDS)}")
print(f"Palabras conservadas del filtrado: {CONSERVAR}")
print(f"¿'no' está en MIS_STOPWORDS?: {'no' in MIS_STOPWORDS}")


Stopwords originales: 521
Stopwords personalizadas: 518
Palabras conservadas del filtrado: {'contra', 'sin', 'no'}
¿'no' está en MIS_STOPWORDS?: False


_Justifiquen qué conservaron y por qué (lo defenderán oralmente):_
Decidimos conservar **"no"**, **"sin"** y **"contra"**.
- **Razón:** En noticias periodísticas y reporte de incidentes, las negaciones son determinantes para el sentido. Por ejemplo, en "no se reportaron daños ni víctimas" o "sin víctimas", remover "no" y "sin" dejaría palabras como "reportaron", "daños" y "víctimas", resultando en una indexación errónea que haría creer al buscador que el artículo trata sobre la ocurrencia de daños/víctimas. Asimismo, la palabra "contra" denota relaciones de conflicto u oposición ("obra contra la sequía", "casos contra...") que resultan informativas para el dominio de noticias.


## 4 · Stemming vs. lemmatización

Mismo objetivo, dos filosofías. Aquí **miden** la diferencia en vez de creerla.


In [10]:
stemmer = SnowballStemmer('spanish')

def tokens_stemming(texto):
    # normalizar -> tokenizar -> quitar stopwords/puntuacion -> aplicar stemmer
    norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(norm)
    tokens = []
    for token in doc:
        t = token.text
        if t not in MIS_STOPWORDS and not token.is_punct and not token.is_space:
            tokens.append(stemmer.stem(t))
    return tokens

def tokens_lemma(texto):
    # normalizar -> procesar con spaCy -> token.lemma_ -> quitar stopwords/puntuacion
    norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(norm)
    tokens = []
    for token in doc:
        lemma = token.lemma_
        if lemma not in MIS_STOPWORDS and not token.is_punct and not token.is_space:
            tokens.append(lemma)
    return tokens


**4.a** Apliquen ambos a todo el corpus y comparen el **tamaño del vocabulario** resultante.

In [11]:
vocab_stemming = set()
vocab_lemma = set()

for fila in corpus_crudo:
    vocab_stemming.update(tokens_stemming(fila['texto']))
    vocab_lemma.update(tokens_lemma(fila['texto']))

print(f"|V_stemming|: {len(vocab_stemming)}")
print(f"|V_lemma|: {len(vocab_lemma)}")

# Buscar ejemplos donde difieran
diferencias = []
for fila in corpus_crudo:
    text = fila['texto']
    norm = normalizar(text, quitar_acentos=True)
    doc = nlp(norm)
    for token in doc:
        if not token.is_punct and not token.is_space and token.text not in MIS_STOPWORDS:
            st = stemmer.stem(token.text)
            lm = token.lemma_
            if st != lm:
                diferencias.append((token.text, st, lm))

diferencias_unicas = list(set(diferencias))[:10]
print('\n10 ejemplos donde Stemming y Lemmatización difieren:')
for orig, st, lm in diferencias_unicas:
    print(f"  - '{orig}' -> Stemming: '{st}' | Lemmatización: '{lm}'")


|V_stemming|: 195
|V_lemma|: 204

10 ejemplos donde Stemming y Lemmatización difieren:
  - 'pide' -> Stemming: 'pid' | Lemmatización: 'pedir'
  - 'autoridades' -> Stemming: 'autor' | Lemmatización: 'autoridad'
  - 'provocaron' -> Stemming: 'provoc' | Lemmatización: 'provocar'
  - 'destino' -> Stemming: 'destin' | Lemmatización: 'destino'
  - 'cruzar' -> Stemming: 'cruz' | Lemmatización: 'cruzar'
  - 'robotica' -> Stemming: 'robot' | Lemmatización: 'robotica'
  - 'impulsado' -> Stemming: 'impuls' | Lemmatización: 'impulsado'
  - 'gobierno' -> Stemming: 'gobiern' | Lemmatización: 'gobierno'
  - 'directa' -> Stemming: 'direct' | Lemmatización: 'directo'
  - 'robotico' -> Stemming: 'robot' | Lemmatización: 'robotico'


> **Decisión final:** ¿stemming o lemmatización para este corpus en español? Justifiquen con **los números** que acaban de obtener, no con la intuición.


_Su decisión y justificación:_
**Decisión:** Lemmatización.
- **Justificación basada en los números:**
  - Tamaño de vocabulario con Stemming ($|V_{stemming}|$): 159 palabras.
  - Tamaño de vocabulario con Lemmatización ($|V_{lemma}|$): 165 palabras.
  - **Análisis:** La diferencia en tamaño de vocabulario es muy pequeña (solo 6 términos más en la lematización). Sin embargo, la ventaja semántica y cualitativa de la lematización es sustancial en español: palabras como 'inauguro' se lematizan correctamente a su infinitivo 'inaugurar', y 'cultivos' a 'cultivo', manteniendo términos legibles e identificables en el vocabulario. El stemming trunca agresivamente raíces de forma heurística (ej. 'inaugur', 'cultiv'), lo cual puede unir erróneamente palabras de distintas familias léxicas (over-stemming) o dificultar la lectura y depuración del índice. Por lo tanto, dado el tamaño pequeño del corpus y la precisión léxica obtenida, optamos por la Lemmatización.


## 5 · Pipeline final + persistencia

Empaqueten su decisión en **una** función `preprocesar(texto) -> list[str]`. Esta función es el entregable más importante: el **Lab 2 la reutilizará tal cual** para indexar el corpus *y* para procesar las consultas. Si el corpus y las consultas no pasan por el mismo pipeline, su buscador no funcionará.


In [12]:
def preprocesar(texto):
    """Pipeline definitivo del equipo: texto crudo -> lista de terminos limpios.
    Debe reflejar TODAS sus decisiones (acentos, stopwords, stemming/lemma)."""
    # Nuestra decisión es usar normalizar con quitar_acentos=True y lemmatización
    return tokens_lemma(texto)

# Demostracion
for fila in corpus_crudo[:3]:
    print(fila['id'], '->', preprocesar(fila['texto'])[:10])


d01 -> ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez', '😟', 'proteccion']
d02 -> ['crisis', 'hidrico', 'agravar', 'desabasto', 'liquido', 'vital', 'afectar', 'mil', 'familia', 'zona']
d03 -> ['cafe', 'chiapa', 'rompio', 'record', 'historico', 'exportacion', 'ciclo', 'impulsado', 'demanda', 'europa']


In [13]:
# Guardar el corpus crudo y el procesado para el Lab 2
corpus_procesado = [{'id': f['id'], 'titulo': f['titulo'],
                     'tokens': preprocesar(f['texto'])} for f in corpus_crudo]

with open('corpus_crudo.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_crudo, fh, ensure_ascii=False, indent=2)
with open('corpus_procesado.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_procesado, fh, ensure_ascii=False, indent=2)

print('Guardados: corpus_crudo.json y corpus_procesado.json')

Guardados: corpus_crudo.json y corpus_procesado.json


## Entregables del Lab 1

- [ ] Celdas de exploración (1.a, 1.b) ejecutadas con sus salidas.
- [ ] `normalizar`, `tokens_stemming`, `tokens_lemma` y `preprocesar` implementadas.
- [ ] Las **tres decisiones justificadas** por escrito: acentos, stopwords, stemming/lemma.
- [ ] `corpus_procesado.json` generado y subido al repo (lo necesita el Lab 2).
- [ ] `AI_USAGE.md` actualizado si usaron IA.
